> ### AI Lab (Post-lab Report)

# Lab \#9: Handling Missing Data

<h4>
<div style="text-align: right">Due date: November 19, 2026, 2:00 PM (KST).</div><br>
<div style="text-align: right">Please upload [AILAB_w9_ID_Name.ipynb] to ONLINE CLASSROOM.</div><br>
<div style="text-align: right">For <b>Name</b>, using the team leader's name is recommended.</div>
</h4>

### Team Members

**Please write your student ID and name below.**

<h2><span style="color:blue">
[2026000000] [홍길동]<br> [2026000001] [김영희]<br> [2026000002] [김철수]
</span></h2>

<details>
<summary><b>📌 Instructions / Submission (CLICK!)</b></summary>
<ul>
<li>Train a GRAPE imputer separately at <b>10%, 30%, 50%, 70%, and 90% missingness</b>.</li>
<li>Use the supplied <b>five row folds</b>: train a fresh model for each rate/fold pair, totaling <b>25 fits</b>. Compare mean and standard deviation of held-out RMSE and MAE and plot the trend.</li>
<li>Experiment freely with architecture, loss and training strategy. No additional experiment types or minimum counts are prescribed.</li>
<li>The highest priority is improving GRAPE performance. Final scoring uses average MAE gains over the best competing baseline across the five missing rates. Held-out fold targets must never influence training or selection.</li>
<li>Submit one executed <code>AILAB_w9_ID_Name.ipynb</code> including results, the missing-rate trend and Results Report. Deadline and grading follow the course announcement.</li>
</ul>

</details>

<details>
<summary><b>📌 안내사항 / 제출 (CLICK!, 위와 같은 내용)</b></summary>
<ul>
<li>결측률 <b>10%, 30%, 50%, 70%, 90%</b> 각각에서 GRAPE 기반 결측값 복원 모델을 학습합니다.</li>
<li>동일한 <b>행 단위 5-fold</b>를 사용하여 결측률·fold 조합마다 새 모델을 학습합니다(<b>총 25회</b>). 평가 RMSE·MAE의 평균과 표준편차 및 결측률에 따른 추이를 비교합니다.</li>
<li>모델 구조, Loss, 학습 방법 등은 자유롭게 변경할 수 있습니다. 추가 필수 실험 종류나 최소 횟수는 없습니다.</li>
<li><b>GRAPE의 성능을 높이는 것이 최우선 목표</b>입니다. 최종 점수는 결측률별 최우수 baseline 대비 <b>MAE 개선율의 평균</b>입니다. 평가 fold의 정답은 학습·설정 선택에 사용하지 않습니다.</li>
<li>결과, 결측률별 추이 그래프와 결과보고서를 포함한 <code>AILAB_w9_ID_Name.ipynb</code>를 제출하세요. 마감일과 배점은 수업 공지를 따릅니다.</li>
</ul>

</details>

<details>
<summary><b>📋 Five-Fold Protocol / 5-fold 평가 방법 (CLICK!)</b></summary>

- Use the eight Concrete features; exclude the original compressive-strength target.
- Partition sample rows into five shuffled, fixed folds (seed 42). Each fold is held out once at each missing rate. Reinitialize the model and optimizer for every rate/fold combination.
- Use a fixed random cell ordering **within each feature**. Hide 10/30/50/70/90% of the rows in each feature, with nested masks. The same mask for a rate is shared across folds. Actual rates within an individual train/test fold can differ slightly from the overall rate; record both.
- During training and validation, the graph contains **training rows only**, and only their observed cells form edges. Fit normalization on these observed training values only. Randomly dropped observed training edges provide training supervision; the naturally masked training cells provide validation targets for checkpoint selection only.
- Evaluate only masked cells in held-out rows. For inference, build a combined context graph from observed cells in training and held-out rows. Held-out observed values are legitimate imputation inputs; held-out hidden values never appear in the graph. This is **held-out-row imputation with observed test-time graph context**, not prediction for wholly unobserved rows or a claim of strict isolated-row inference.
- During checkpoint selection, validation targets are excluded from gradient updates and input edges. Settings and training duration must be fixed before any optional refit within the outer training partition. Held-out evaluation targets remain excluded throughout, and evaluation masks must remain unchanged. The supplied loop uses validation-selected checkpoints without refitting; all held-out scores are revealed in the final evaluation section.
- Report **normalized-cell MAE** (primary error metric) and RMSE for each fold, plus the mean and sample standard deviation over five folds. Normalization is fitted independently per fold/rate, so its range can vary; interpret this limitation when comparing rates. These normalized errors are not physical-unit errors.
- The training edge-drop probability is separate from the requested missing rate. It is a freely editable learning strategy.
</details>

In [ ]:
import datetime
print("This code is written at " + str(datetime.datetime.now()))

## 1. Environment, Original Source and Automatic Data Download

Install PyTorch/PyTorch Geometric and the `torch-scatter` wheel matching your PyTorch build. The dependency cell detects the PyTorch/CUDA version and downloads a binary wheel from the official PyG wheel index. Internet is required on first run. If you changed an already loaded package version, restart the kernel and run from the top.

The next source cell downloads unchanged official GRAPE files at a pinned commit, verifies every file hash and imports them directly. No local model reimplementation is used.

In [ ]:
%pip -q install "torch>=2,<3" "numpy>=1.24,<3" "pandas>=2,<3" "matplotlib>=3.7,<4" "torch-geometric==2.7.0"

In [ ]:
# INSTALL ORIGINAL GRAPE EXTENSION DEPENDENCY.
import sys
import subprocess
import importlib.util
import torch

if importlib.util.find_spec("torch_scatter") is None:
    torch_parts = torch.__version__.split("+")[0].split(".")
    torch_wheel_version = ".".join(torch_parts[:2]) + ".0"
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_wheel_version}+{cuda_tag}.html"
    print("Installing torch-scatter from:", wheel_url)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--only-binary=:all:",
                           "--no-deps", "torch-scatter", "-f", wheel_url])
import torch_geometric
import torch_scatter
print("torch", torch.__version__, "PyG", torch_geometric.__version__,
      "torch-scatter", torch_scatter.__version__)

In [ ]:
from pathlib import Path
import copy
import hashlib
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display

SOURCE_COMMIT = "0ea0c59272a977d0184a8fd95178f68211455ef5"
MASK_SEED = 42  # Fixed evaluation masks
DATA_ROOT = Path("dataset/grape_concrete")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
device = torch.device("cpu")  # Editable training setting
# Small full-batch graphs generally need few CPU threads.
torch.set_num_threads(2)
print("torch:", torch.__version__, "numpy:", np.__version__, "device:", device)

In [ ]:
# DOWNLOAD AND IMPORT UNMODIFIED OFFICIAL GRAPE SOURCE.
import sys
import inspect
SOURCE_ROOT = Path("external/GRAPE") / SOURCE_COMMIT
SOURCE_SHA256 = {'models/gnn_model.py': 'f7366b518221d36a1ad4a1fbcf6ebc2f07759c9a80d550c64481e35d97fc14d7', 'models/egsage.py': 'ee9f47babccf45e5947163151cad56bef1752d1b444745b3548197f3c0e36d34', 'models/egcn.py': '04f7f323e309690bdee47fb9d91d62b38bec2392ea48507e200a74cb4a05d61a', 'models/prediction_model.py': '18ae200239b91b68eb07c7ecaf789aa7a560dd77af8148d5f82558765ad16a3b', 'utils/utils.py': 'acbd0c5e75435eb6c414317547779a515a4e5df097c1bfc569332533ce477ad1', 'training/gnn_mdi.py': 'ab1ff518d4cb7a7e578431efab86161a3d8827a296e0eecac73372a715d71f3b', 'LICENSE': 'd637f90687988ca2e10a7dd6a4385e64bcf8ef1f4d5cc9a40a91470206e6ee79'}
for relative, expected_hash in SOURCE_SHA256.items():
    path = SOURCE_ROOT / relative
    if not path.exists():
        url = f"https://raw.githubusercontent.com/maxiaoba/GRAPE/{SOURCE_COMMIT}/{relative}"
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=120) as response:
            payload = response.read()
        assert hashlib.sha256(payload).hexdigest() == expected_hash, relative
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(payload)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash, (
        f"Upstream source changed locally: {path}. Use a fresh source folder for the original baseline.")
sys.path.insert(0, str(SOURCE_ROOT.resolve()))
from models.gnn_model import GNNStack
from models.egsage import EGraphSage
from models.prediction_model import MLPNet
for cls in [GNNStack, EGraphSage, MLPNet]:
    location = Path(inspect.getfile(cls)).resolve()
    assert SOURCE_ROOT.resolve() in location.parents, f"Conflicting module import: {location}"
    print("Original class:", cls.__name__, "from", location)
print("Verified original source at commit:", SOURCE_COMMIT)

In [ ]:
# DOWNLOAD ORIGINAL CONCRETE DATA — no manual upload required.
DATA_URL = (f"https://raw.githubusercontent.com/maxiaoba/GRAPE/{SOURCE_COMMIT}/"
            "uci/raw_data/concrete/data/data.txt")
DATA_FILE = DATA_ROOT / "data.txt"
if not DATA_FILE.exists():
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        payload = response.read()
    # Check parsing before saving, so an interrupted download is not cached.
    import io
    parsed = np.loadtxt(io.BytesIO(payload))
    assert parsed.shape == (1030, 9) and np.isfinite(parsed).all()
    DATA_FILE.write_bytes(payload)
    print("Downloaded:", DATA_FILE)
else:
    print("Using cached:", DATA_FILE)
raw_data = np.loadtxt(DATA_FILE)
assert raw_data.shape == (1030, 9) and np.isfinite(raw_data).all()
X_complete = raw_data[:, :-1].astype(np.float32)  # Exclude original target.
N_SAMPLES, N_FEATURES = X_complete.shape
DATA_SHA256 = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
print("Feature matrix:", X_complete.shape, "| Original target excluded")
print("Data SHA256:", DATA_SHA256)

## 2. Fixed Folds and Missing-Rate Masks

Five row folds are shared across all missing rates. Per-feature missing masks are nested, so increasing missingness hides additional cells. The complete matrix is retained only to simulate known ground truth; never expose masked values to the graph or normalization.

In [ ]:
MISSING_RATES = [0.10, 0.30, 0.50, 0.70, 0.90]
N_FOLDS = 5
rng = np.random.default_rng(MASK_SEED)
fold_rows = np.array_split(rng.permutation(N_SAMPLES), N_FOLDS)
assert sorted(np.concatenate(fold_rows).tolist()) == list(range(N_SAMPLES))
feature_orders = np.stack([rng.permutation(N_SAMPLES) for _ in range(N_FEATURES)], axis=1)
missing_masks = {}
for rate in MISSING_RATES:
    mask = np.zeros_like(X_complete, dtype=bool)
    for j in range(N_FEATURES):
        mask[feature_orders[:int(round(rate * N_SAMPLES)), j], j] = True
    missing_masks[rate] = mask
for lower, higher in zip(MISSING_RATES[:-1], MISSING_RATES[1:]):
    assert np.all(~missing_masks[lower] | missing_masks[higher])
display(pd.DataFrame([{"Missing rate": rate, "Actual overall missing": mask.mean(),
                       "Hidden cells": int(mask.sum()), "Observed cells": int((~mask).sum())}
                     for rate, mask in missing_masks.items()]))
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, rate in zip(axes, MISSING_RATES):
    ax.imshow(~missing_masks[rate][:60], aspect="auto", cmap="Greys", interpolation="nearest")
    ax.set(title=f"Missing {rate:.0%}", xlabel="Feature")
axes[0].set_ylabel("First 60 rows; dark = observed")
plt.tight_layout()
plt.show()

## 3. Original GRAPE Model and Fold Data

The wrapper below passes graph data into the original `get_gnn` and `MLPNet`; it defines no new graph layers. The original source is unchanged. The fold-data and evaluation helpers implement this lab's evaluation protocol. Model and training settings remain editable for your own experiments.

In [ ]:
# ORIGINAL MODEL IMPLEMENTATIONS — downloaded unchanged and verified above.
from models.gnn_model import get_gnn
from models.prediction_model import MLPNet
from utils.utils import get_known_mask
from types import SimpleNamespace

class OriginalGRAPE(nn.Module):
    """Interface wrapper only: all GNN and decoder layers come from upstream GRAPE."""
    def __init__(self, input_dim, config):
        super().__init__()
        args = SimpleNamespace(
            model_types="_".join(["EGSAGE"] * config["layers"]),
            norm_embs=None, post_hiddens=None, node_dim=config["hidden_dim"],
            edge_dim=config["edge_dim"], edge_mode=1, dropout=0.0,
            gnn_activation="relu", concat_states=False, aggr="mean")
        self.gnn = get_gnn(SimpleNamespace(num_node_features=input_dim, edge_attr_dim=1), args)
        self.impute_model = MLPNet(2 * config["hidden_dim"], 1,
            hidden_layer_sizes=[config["hidden_dim"]], hidden_activation="relu", dropout=0.0)

    def forward(self, nodes, edge_index, edges, query_pairs):
        embeddings = self.gnn(nodes, edges, edge_index)
        return self.impute_model([
            embeddings[query_pairs[0]], embeddings[query_pairs[1]]])[:, 0]

def reconstruction_metrics(predictions, targets):
    assert predictions.shape == targets.shape and torch.isfinite(predictions).all()
    return {"RMSE": float(torch.sqrt(F.mse_loss(predictions, targets)).item()),
            "MAE": float(F.l1_loss(predictions, targets).item())}


In [ ]:
def make_graph_data(matrix, missing, minimum, value_range):
    normalized = (matrix - minimum) / value_range
    n_rows, n_features = matrix.shape
    rows, cols = np.where(~missing)
    observed_pairs = torch.tensor(np.stack([rows, n_rows + cols]), dtype=torch.long, device=device)
    observed_values = torch.tensor(normalized[rows, cols], dtype=torch.float32, device=device)
    hidden_rows, hidden_cols = np.where(missing)
    query_pairs = torch.tensor(np.stack([hidden_rows, n_rows + hidden_cols]), dtype=torch.long, device=device)
    nodes = torch.cat([torch.ones(n_rows, n_features), torch.eye(n_features)]).to(device)
    return dict(nodes=nodes, pairs=observed_pairs, values=observed_values,
                queries=query_pairs, hidden_rows=hidden_rows, hidden_cols=hidden_cols,
                normalized=normalized)

def observed_graph(data, keep=None):
    pairs, values = data["pairs"], data["values"]
    if keep is not None:
        pairs, values = pairs[:, keep], values[keep]
    return torch.cat([pairs, pairs.flip(0)], dim=1), torch.cat([values[:, None], values[:, None]])

def make_fold_data(rate, fold):
    test_rows = fold_rows[fold]
    training_rows = np.concatenate([rows for i, rows in enumerate(fold_rows) if i != fold])
    missing = missing_masks[rate]
    training_matrix, training_missing = X_complete[training_rows], missing[training_rows]
    observed = np.where(training_missing, np.nan, training_matrix)
    assert np.all(np.isfinite(observed).sum(axis=0) >= 2)
    minimum = np.nanmin(observed, axis=0)
    value_range = np.nanmax(observed, axis=0) - minimum
    value_range = np.where(value_range > 0, value_range, 1.0)
    train_data = make_graph_data(training_matrix, training_missing, minimum, value_range)
    # These hidden training-row cells are validation targets, never graph inputs.
    val_targets = torch.tensor(train_data["normalized"][training_missing], dtype=torch.float32, device=device)
    normalized_observed = (observed - minimum) / value_range
    means = np.nanmean(normalized_observed, axis=0)
    medians = np.nanmedian(normalized_observed, axis=0)
    return dict(train=train_data, val_targets=val_targets, minimum=minimum,
                value_range=value_range, means=means, medians=medians, training_rows=training_rows,
                test_rows=test_rows, training_missing=training_missing)

## 4. Train at Each Missing Rate with 5-Fold CV

The loop performs **25 independent fits**. Epochs, architecture, loss and training strategy are editable. Checkpoints are selected by normalized Validation MAE using hidden cells in training rows. The held-out fold is not evaluated until the checkpoint is frozen. Training edge dropout is independent of the 10/30/50/70/90% evaluation masks.

**결측률별 5개 fold에서 매번 새 모델을 학습합니다.** 추가 실험 종류나 횟수는 자유이며, 이 비교에 필요한 25개 학습은 아래 반복문에서 자동 수행합니다.

The supplied baseline uses original `loss_mode=0` (MSE over all observed training cells) and 64-dimensional embeddings. `loss_mode=1` is also supported. The model/decoder source and masking helper are upstream; the five-fold orchestration is lab-specific.

### 4.1 Optional Tuning Guide / 선택적 튜닝 가이드

**These are optional ideas, not required experiments.** Keep the original GRAPE model family and choose your own approach. No particular item or number of tuning experiments is required beyond the five missing rates and five folds already specified.

**아래 항목은 선택적으로 참고하는 예시입니다.** 원본 GRAPE 모델 계열을 유지하면서 자유롭게 실험하세요. 앞서 지정한 결측률 5종 × 5-fold 비교 외에 특정 튜닝 항목이나 최소 실험 횟수를 요구하지 않습니다.

| Item / 항목 | Possible changes / 변경해 볼 내용 | What to examine / 확인할 점 |
|:--|:--|:--|
| Training duration / 학습 횟수 | Epoch count, early stopping / Epoch 수, 조기 종료 | Has training converged? Is Validation performance deteriorating? / 충분히 수렴했는지, Validation 성능이 나빠지기 시작하는지 |
| Learning rate / 학습률 | Learning rate and scheduler / 학습률과 Scheduler | Convergence speed and stability / 수렴 속도와 안정성 |
| Model size / 모델 크기 | Node/edge embedding dimensions, EGSAGE layer count, decoder width / Node·Edge 임베딩 차원, EGSAGE 계층 수, Decoder 크기 | Whether added capacity improves Validation performance or increases overfitting / 표현력 증가가 개선 또는 과적합으로 이어지는지 |
| Edge dropout / 간선 Dropout | `edge_keep_probability` | How much observed training context to retain / 학습 중 관측 간선을 얼마나 유지할지 |
| Loss / 손실 함수 | MSE, MAE, Huber, or a mixed loss / MSE, MAE, Huber 또는 혼합 손실 | Changes in both RMSE and MAE; improving one need not improve the other / 두 지표가 함께 개선되는지, 상충하는지 |
| Regularization / 정규화 | Weight decay and model dropout / Weight decay, 모델 Dropout | Generalization on Validation cells / Validation 셀에서의 일반화 성능 |
| Training strategy / 학습 전략 | Optimizer, initialization, or a suitable batching strategy / Optimizer, 초기화, 적절한 Batch 구성 | Stable learning while preserving graph context and excluding hidden targets / 그래프 문맥과 정답 분리를 유지하면서 안정적으로 학습하는지 |

**Missing rate is different from training edge dropout.** A missing rate of 90% leaves only 10% of cells observed. `edge_keep_probability` then controls how many of those observed training edges remain visible during an update. A high missing rate does not imply that aggressively dropping the remaining edges is helpful.

**데이터 결측률과 학습 중 간선 Dropout은 서로 다른 설정입니다.** 결측률이 90%라면 관측 셀은 10%만 남습니다. `edge_keep_probability`는 그 관측 간선 중 학습 단계에서 유지할 비율입니다. 결측률이 높다고 해서 남은 관측 간선도 많이 가리는 것이 유리하다는 뜻은 아닙니다.

**Use Validation only to choose settings, losses, stopping criteria and checkpoints.** The supplied selector minimizes Validation MAE; if you change the selection criterion, define it using Validation and document it before held-out evaluation. Do not tune against held-out fold scores. If you refit within the outer training partition after selecting settings, do not reuse training targets as an independent validation score; keep held-out evaluation targets hidden. Keep folds, missing masks, preprocessing isolation and reported metric definitions unchanged.

**설정·Loss·중단 기준·체크포인트는 각 fold의 Validation으로만 선택하세요.** 기본 코드는 Validation MAE를 최소화합니다. 선택 기준을 변경한다면 Validation으로 정의하고 평가 전에 보고서에 명시하세요. 평가 fold 점수로 튜닝하면 안 됩니다. 설정 선택 후 바깥쪽 학습 구간 내에서 재학습한다면, 학습에 사용한 값을 독립 Validation 점수로 재사용하지 말고 평가 fold의 정답은 계속 숨겨 두세요. 분할·결측 마스크·전처리의 학습 구간 분리·평가 지표 정의는 유지하세요.

The existing `CONFIG` exposes some settings; other changes may require editing the wrapper or training loop. Continue to use upstream GRAPE classes rather than replacing them with another model family.

일부 항목은 `CONFIG`에서 바꿀 수 있고, 나머지는 원본 GRAPE 클래스를 호출하는 wrapper나 학습 반복문 수정이 필요할 수 있습니다. 다른 모델 계열로 대체하지 않고 원본 GRAPE 클래스를 사용하세요.

**Improving GRAPE performance is the highest priority.** Maximize average MAE gains over the strongest Mean/Median baseline across all five missing rates. Report measured results honestly; worse-than-baseline cases reduce the final score.

**GRAPE의 성능을 높이는 것이 최우선 목표입니다.** 5개 결측률에서 최우수 Mean/Median baseline 대비 MAE 개선율 평균을 최대화하세요. Baseline보다 나쁜 결과도 숨기거나 제외하지 않으며, 해당 조건의 음수 개선율도 평균에 반영됩니다.

In [ ]:
CONFIG = dict(hidden_dim=64, edge_dim=64, layers=3, epochs=150, learning_rate=0.001,
              weight_decay=0.0, edge_keep_probability=0.7, train_seed=42, loss_mode=0)
assert CONFIG["epochs"] >= 1 and 0 < CONFIG["edge_keep_probability"] < 1
OUTPUT_ROOT = Path("results/grape_missing_5fold")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def training_loss(predictions, targets):
    # Freely change the reconstruction loss.
    return F.mse_loss(predictions, targets)

def fit_one_fold(data, seed):
    torch.manual_seed(seed)
    graph = data["train"]
    model = OriginalGRAPE(N_FEATURES, CONFIG).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"],
                                 weight_decay=CONFIG["weight_decay"])
    full_index, full_attr = observed_graph(graph)
    best_state, best_mae, best_epoch = None, float("inf"), None
    history = []
    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        keep = get_known_mask(CONFIG["edge_keep_probability"], len(graph["values"])).to(device)
        edge_index, edge_attr = observed_graph(graph, keep)
        optimizer.zero_grad()
        # Original training/gnn_mdi.py: predict all training cells (loss_mode=0).
        pred = model(graph["nodes"], edge_index, edge_attr, graph["pairs"])
        if CONFIG["loss_mode"] == 1:
            # Match original loss_mode=1: zero error on kept cells, mean over all cells.
            pred = torch.where(keep, graph["values"], pred)
        loss = training_loss(pred, graph["values"])
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(graph["nodes"], full_index, full_attr, graph["queries"])
            metrics = reconstruction_metrics(val_pred, data["val_targets"])
        history.append({"Epoch": epoch, "Train loss": loss.item(), "Validation MAE": metrics["MAE"]})
        if metrics["MAE"] < best_mae:
            best_state = copy.deepcopy(model.state_dict())
            best_mae, best_epoch = metrics["MAE"], epoch
    assert best_state is not None
    model.load_state_dict(best_state)
    return model.eval(), best_epoch, best_mae, history

def evaluate_heldout_fold(model, data, rate):
    # Inference only: observed held-out values may provide graph context.
    rows = np.r_[data["training_rows"], data["test_rows"]]
    missing = missing_masks[rate][rows]
    inference = make_graph_data(X_complete[rows], missing, data["minimum"], data["value_range"])
    split = len(data["training_rows"])
    query_is_test = inference["hidden_rows"] >= split
    assert np.all(np.isin(rows[inference["hidden_rows"][query_is_test]], data["test_rows"]))
    query = inference["queries"][:, query_is_test]
    target = torch.tensor(inference["normalized"][missing][query_is_test], dtype=torch.float32, device=device)
    edge_index, edge_attr = observed_graph(inference)
    with torch.no_grad():
        pred = model(inference["nodes"], edge_index, edge_attr, query)
    mean_pred = torch.tensor(data["means"][inference["hidden_cols"][query_is_test]],
                             dtype=torch.float32, device=device)
    median_pred = torch.tensor(data["medians"][inference["hidden_cols"][query_is_test]],
                               dtype=torch.float32, device=device)
    return (reconstruction_metrics(pred, target), reconstruction_metrics(mean_pred, target),
            reconstruction_metrics(median_pred, target), pred, target)

In [ ]:
# TRAIN ALL 25 MODELS — no held-out fold scores are computed here.
learning_records, training_records, trained_checkpoints = [], [], []
for rate in MISSING_RATES:
    for fold in range(N_FOLDS):
        data = make_fold_data(rate, fold)
        seed = CONFIG["train_seed"] + fold
        model, epoch, val_mae, history = fit_one_fold(data, seed)
        learning_records.extend([{**row, "Missing rate": rate, "Fold": fold + 1} for row in history])
        training_records.append({"Missing rate": rate, "Fold": fold + 1,
                                 "Selected epoch": epoch, "Validation MAE": val_mae})
        checkpoint_path = OUTPUT_ROOT / f"grape_missing{int(rate*100):02d}_fold{fold+1}.pt"
        torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "config": copy.deepcopy(CONFIG), "missing_rate": rate, "fold": fold + 1,
            "selected_epoch": epoch, "validation_mae": val_mae,
            "training_rows": torch.tensor(data["training_rows"]), "test_rows": torch.tensor(data["test_rows"]),
            "missing_mask": torch.tensor(missing_masks[rate]), "mask_seed": MASK_SEED,
            "minimum": torch.tensor(data["minimum"]), "range": torch.tensor(data["value_range"]),
            "source_commit": SOURCE_COMMIT, "source_sha256": SOURCE_SHA256,
            "torch_version": str(torch.__version__), "pyg_version": torch_geometric.__version__,
            "data_sha256": DATA_SHA256}, checkpoint_path)
        trained_checkpoints.append(checkpoint_path)
        print(f"Missing {rate:.0%} | Fold {fold+1}/5 | Epoch {epoch} | Validation MAE {val_mae:.5f}", flush=True)
        pd.DataFrame(learning_records).to_csv(OUTPUT_ROOT / "learning_history.csv", index=False)
learning_history = pd.DataFrame(learning_records)
assert len(trained_checkpoints) == 25
display(pd.DataFrame(training_records))
print("All models frozen. Run final evaluation only after completing validation-based selection.")


## 5. Final Quantitative Evaluation: Average MAE Gains

**Improving GRAPE performance is the highest priority. / GRAPE의 성능을 높이는 것이 최우선 목표입니다.**

Run this section after all settings and 25 checkpoints are frozen. It reveals held-out-fold results for GRAPE, Mean and Median and reports five-fold means and sample standard deviations. Do not use these results to revise settings.

For each missing rate $r$, let $G_r$ be GRAPE's mean MAE over five folds, and define the strongest competing baseline as:

$$B_r=\min(\overline{\mathrm{MAE}}_{\mathrm{Mean},r},\overline{\mathrm{MAE}}_{\mathrm{Median},r}).$$

$$\mathrm{Gain}_r(\%)=100\frac{B_r-G_r}{B_r},\qquad
\boxed{\mathrm{Average\ Gains}=\frac{1}{5}\sum_{r\in\{10,30,50,70,90\}\%}\mathrm{Gain}_r(\%)}.$$

**Runner-up reference:** when GRAPE ranks first, this baseline is the runner-up. If GRAPE does not rank first, still compare it with the **best competing baseline**; the gain is negative. Do not switch to a weaker baseline, clip negative gains, or exclude missing rates. Baseline selection uses the five-fold mean MAE, not the best baseline chosen separately in each fold. The final score is the **average of five percentage gains**. If the reference MAE is zero, relative gain is undefined and the cell stops rather than inventing a score.

결측률마다 Mean·Median 중 5-fold 평균 MAE가 작은 방법을 비교 기준으로 정합니다. GRAPE가 1등이면 이 방법이 runner-up이며, GRAPE가 1등이 아니어도 최우수 baseline을 기준으로 계산해 음수 개선율을 반영합니다. 5개 개선율의 **평균이 최종 점수**이며 높을수록 좋습니다. 먼저 fold별 MAE를 평균한 뒤 개선율을 계산합니다.

The same fixed masks and fold-specific normalization are used for all methods. Mean/Median statistics use observed training-row values only. The graphs show mean ± sample standard deviation, not confidence intervals. These are five-fold held-out results, not a separate untouched final test set if repeatedly used for tuning.

In [ ]:
# FINAL HELD-OUT EVALUATION — run after freezing all 25 checkpoints.
fold_records = []
for checkpoint_path in trained_checkpoints:
    saved = torch.load(checkpoint_path, map_location=device, weights_only=True)
    rate, fold = saved["missing_rate"], saved["fold"] - 1
    data = make_fold_data(rate, fold)
    assert torch.equal(saved["missing_mask"].cpu(), torch.tensor(missing_masks[rate]))
    assert np.array_equal(saved["training_rows"].cpu().numpy(), data["training_rows"])
    assert np.array_equal(saved["test_rows"].cpu().numpy(), data["test_rows"])
    assert np.array_equal(saved["minimum"].cpu().numpy(), data["minimum"])
    assert np.array_equal(saved["range"].cpu().numpy(), data["value_range"])
    model = OriginalGRAPE(N_FEATURES, saved["config"]).to(device)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    grape_metrics, mean_metrics, median_metrics, predictions, targets = evaluate_heldout_fold(model, data, rate)
    for label, metrics in [("GRAPE", grape_metrics), ("Mean baseline", mean_metrics), ("Median baseline", median_metrics)]:
        fold_records.append({"Missing rate": rate, "Fold": fold + 1, "Model": label,
            "RMSE": metrics["RMSE"], "MAE": metrics["MAE"],
            "Train missing actual": data["training_missing"].mean(),
            "Test missing actual": missing_masks[rate][data["test_rows"]].mean(),
            "Test hidden cells": len(targets)})
fold_results = pd.DataFrame(fold_records)
assert len(fold_results) == 75
assert not fold_results.duplicated(["Missing rate", "Fold", "Model"]).any()
fold_results.to_csv(OUTPUT_ROOT / "fold_results.csv", index=False)
_final_grape_cv_evaluated = True

summary = fold_results.groupby(["Model", "Missing rate"]).agg(
    RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
    MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"), Folds=("Fold", "count")
).reset_index()
assert (summary["Folds"] == 5).all()
display(fold_results.round(5))
display(summary.round(5))
summary.to_csv(OUTPUT_ROOT / "missing_rate_summary.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, table in summary.groupby("Model", sort=False):
    table = table.sort_values("Missing rate")
    for ax, metric in zip(axes, ["RMSE", "MAE"]):
        ax.errorbar(table["Missing rate"] * 100, table[f"{metric}_mean"],
            yerr=table[f"{metric}_std"], marker="o", capsize=4, label=label)
        ax.set(xlabel="Missing rate (%)", ylabel=f"Normalized {metric}",
               title=f"5-fold held-out {metric}: mean ± sample SD")
        ax.set_xticks([10, 30, 50, 70, 90])
        ax.grid(alpha=0.2)
        ax.legend()
plt.tight_layout()
plt.show()
# Per-fold trajectories expose variation hidden by the mean.
fig, ax = plt.subplots(figsize=(7, 4))
for fold, table in fold_results.query("Model == 'GRAPE'").groupby("Fold"):
    table = table.sort_values("Missing rate")
    ax.plot(table["Missing rate"] * 100, table["RMSE"], "o-", alpha=0.6, label=f"Fold {fold}")
ax.set(xlabel="Missing rate (%)", ylabel="Normalized RMSE", title="GRAPE: individual fold trends")
ax.set_xticks([10, 30, 50, 70, 90])
ax.legend()
plt.tight_layout()
plt.show()
print("Results and 25 model checkpoints saved under:", OUTPUT_ROOT.resolve())

# FINAL SCORE: use the strongest competing baseline at each missing rate.
mae_table = summary.pivot(index="Missing rate", columns="Model", values="MAE_mean").sort_index()
assert list(mae_table.index) == MISSING_RATES
assert np.isfinite(mae_table.to_numpy()).all()
reference = mae_table[["Mean baseline", "Median baseline"]]
reference_mae = reference.min(axis=1)
if (reference_mae <= 0).any():
    raise ValueError("Relative gain is undefined for a zero baseline MAE.")
gain_results = pd.DataFrame({
    "Missing rate (%)": mae_table.index * 100,
    "GRAPE mean MAE": mae_table["GRAPE"].to_numpy(),
    "Mean baseline MAE": mae_table["Mean baseline"].to_numpy(),
    "Median baseline MAE": mae_table["Median baseline"].to_numpy(),
    "Runner-up / best competitor": reference.idxmin(axis=1).to_numpy(),
    "Reference mean MAE": reference_mae.to_numpy(),
    "GRAPE rank": mae_table.rank(axis=1, method="min")["GRAPE"].to_numpy().astype(int),
    "MAE gain (%)": (100 * (reference_mae - mae_table["GRAPE"]) / reference_mae).to_numpy(),
})
average_gain = float(gain_results["MAE gain (%)"].sum())/5
display(gain_results.round(6))
gain_results.to_csv(OUTPUT_ROOT / "final_mae_gains.csv", index=False)
pd.DataFrame([{"Average MAE gain (average of five percentage gains)": average_gain}]).to_csv(
    OUTPUT_ROOT / "final_score.csv", index=False)
print(f"FINAL AVERAGE MAE GAINS = {average_gain:+.4f} (average of five percentage gains; higher is better)")


## 6. Results Report

- Write discussions here. Analyze the algorithm theoretically and empirically.
- Compare and Discuss the results. Note that it must contain the quantitative and qualitative results.

### References

1. [Official GRAPE repository](https://github.com/maxiaoba/GRAPE).
2. [Handling Missing Data with Graph Representation Learning — project and paper](https://snap.stanford.edu/grape/), You, Ma, Ding, Kochenderfer, and Leskovec, NeurIPS 2020.
3. [Original EGraphSage implementation](https://github.com/maxiaoba/GRAPE/blob/0ea0c59272a977d0184a8fd95178f68211455ef5/models/egsage.py).
4. [Original graph stack](https://github.com/maxiaoba/GRAPE/blob/0ea0c59272a977d0184a8fd95178f68211455ef5/models/gnn_model.py) and [imputation training](https://github.com/maxiaoba/GRAPE/blob/0ea0c59272a977d0184a8fd95178f68211455ef5/training/gnn_mdi.py).
5. [UCI Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength).

<details>
<summary><b>Original GRAPE MIT License (CLICK!)</b></summary>

```text
MIT License

Copyright (c) 2020 Xiaobai Ma, Daisy Yi Ding, Jiaxuan You

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

```
</details>